# Chapter 15 — Term Structure Models: Volatility and Distribution

教材 pp. 178–182；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from volatility_model_calculator import *

Model 3 常数波动退化为 Model 2；p.182解析式的一致性。

In [2]:
from Chapter14.short_rate_calculator import GaussianDrift
model=TimeDependentGaussian(lambda t:.002,lambda t:.01)
assert_allclose(model.discount(.03,4),GaussianDrift(.002,.01).discount(.03,4))
assert_allclose(model.moments(.03,4)['variance'],.0004)

CIR 零期限、正值转换；BK 零冲击的对数均值回复。

In [3]:
cir=CIR(.3,.05,.1)
assert_allclose(cir.discount(.03,0),1)
paths=cir.simulate_uniforms(pd.DataFrame({'path':[.1,.5,.9]}),.03,.1)
assert (paths>=0).all().all()
bk=BlackKarasinski(.5,.05,.2)
out=bk.simulate(pd.DataFrame({'path':[0.]}),.02,1)
expected=np.exp(np.log(.05)+(np.log(.02)-np.log(.05))*np.exp(-.5))
assert_allclose(out.iloc[-1,0],expected)
print(paths)

         path
0.0  0.030000
0.1  0.023813
0.2  0.024342
0.3  0.031520


## 时间波动、对数重组树与Courtadon/BK系数；pp.178–182

In [4]:
model=TimeDependentGaussian(lambda t:.002,lambda t:.0126*np.exp(-.025*t))
assert_allclose(model.moments(.03,10)['variance'],.0126**2*(1-np.exp(-.05*10))/.05)
tree=LognormalRate(.01,.2).binomial_tree(.03,3,.5)
assert_allclose(tree.iloc[1,2],.03*np.exp(.01))
courtadon=Courtadon(.2,.05,.1)
assert_allclose(courtadon.diffusion(0,.03),.003)
bk=TimeDependentBlackKarasinski(lambda t:.2,lambda t:.05,lambda t:.1)
path=bk.simulate(pd.DataFrame({'z':[0.]}),.03,.5)
assert_allclose(path.iloc[-1,0],np.exp(np.log(.03)+.1*(np.log(.05)-np.log(.03))))